# 1 Setup

Use local CARE assets only. Set the CTRG root once if it is not already set:

```bash
export CTRG_DATA_ROOT=/path/to/CTRG
```

In [ ]:
import importlib.util
import json
import os
import re
import sys
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import torch

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
SAMPLE_ID = "1270.nii.gz"
CARE_ROOT = Path(os.environ["CARE_ROOT"]).expanduser() if os.environ.get("CARE_ROOT") else next(
    p for base in (Path.cwd(), *Path.cwd().parents)
    for p in (base, base / "CARE_code_internal") if (p / "model_loading/models.py").is_file()
)
CTRG_DATA_ROOT = Path(os.environ["CTRG_DATA_ROOT"]).expanduser()
IMAGE_PATH = CTRG_DATA_ROOT / "images" / SAMPLE_ID
REPORT_PATH = CTRG_DATA_ROOT / "annotations" / "chest_ctrg_r2gen_nitfy_ori_split.json"
CHECKPOINT = CARE_ROOT / "ckpts" / "care.pth"
TEXT_ASSETS = CARE_ROOT / "model_loading" / "text_assets"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast = (lambda: torch.autocast("cuda", dtype=torch.float16)) if device.type == "cuda" else nullcontext
sys.path.insert(0, str(CARE_ROOT))
print({"device": str(device), "amp": device.type == "cuda"})

# 2 Load CARE

Load the bundled checkpoint and tokenizer from local paths.

In [ ]:
from model_loading.models import get_model, get_tokenizer

model = get_model(device, CHECKPOINT, text_assets_path=TEXT_ASSETS)
tokenizer = get_tokenizer(text_assets_path=TEXT_ASSETS)
model.requires_grad_(False).eval()
print("CARE loaded")

# 3 Load CT and Report

Selected illustrative example from the CTRG test set; not a representative performance estimate.

In [ ]:
with REPORT_PATH.open(encoding="utf-8") as stream:
    annotation = json.load(stream)
sample_key = Path(SAMPLE_ID).name.removesuffix(".nii.gz")
row = next(r for r in annotation["test"] if Path(str(r["id"])).name.removesuffix(".nii.gz") == sample_key)
report_text = f"{row.get('report', '')} {row.get('result', '')}".strip()
spec = importlib.util.spec_from_file_location("care_nifti", CARE_ROOT / "downstream/3_vision_language_tasks/retrieval/datasets/nifti.py")
nifti = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nifti)
dataset_row = {
    "image_path": str(IMAGE_PATH), "text_en": report_text, "report_key": sample_key,
    "patient_key": f"unknown:{sample_key}", "body_part": "chest", "pair_id": sample_key,
}
volume = nifti.NiftiRetrievalDataset([dataset_row], preprocessing="ctrg")[0]["volume"]
assert volume.shape == (1, 112, 336, 336) and torch.isfinite(volume).all()
excerpt = re.sub(r"\s+", " ", report_text)
excerpt = excerpt if len(excerpt) <= 280 else excerpt[:277] + "..."
plt.figure(figsize=(5, 5))
plt.imshow(volume[0, volume.shape[1] // 2], cmap="gray")
plt.title(f"CTRG {sample_key} - central slice")
plt.axis("off")
plt.show()
print({"sample_id": sample_key, "report_excerpt": excerpt, "input_shape": tuple(volume.shape)})

# 4 Image Encoding

Encode the prewindowed CT using retrieval normalization.

In [ ]:
import torch.nn.functional as F

image = ((volume.unsqueeze(0) - 0.449) / 0.226).to(device)
with torch.inference_mode(), autocast():
    image_embedding = F.normalize(model.visual(image)[0].float(), dim=-1)
image_norm = torch.linalg.vector_norm(image_embedding, dim=-1)
assert image_embedding.shape == (1, 1024) and torch.isfinite(image_embedding).all()
assert torch.allclose(image_norm, torch.ones_like(image_norm), atol=1e-5)
print({"shape": tuple(image_embedding.shape), "norm": image_norm.item()})

# 5 Text Encoding

Encode the same composed report with the bundled tokenizer.

In [ ]:
tokens = tokenizer([report_text]).to(device)
with torch.inference_mode(), autocast():
    text_embedding = F.normalize(model.encode_text(tokens, normalize=False).float(), dim=-1)
text_norm = torch.linalg.vector_norm(text_embedding, dim=-1)
assert text_embedding.shape == (1, 1024) and torch.isfinite(text_embedding).all()
assert torch.allclose(text_norm, torch.ones_like(text_norm), atol=1e-5)
print({"shape": tuple(text_embedding.shape), "norm": text_norm.item()})